# TP 1 — Train a small machine-learned force field on ethanol

*M2 Nanochimie — AI for nanochemistry — session 1 of 2 (3 h)*

In session 2 you will use deep neural-network force fields (DNNF) trained on DFT data for SN2 reactions.
Today you build a **mini version** of such a model yourself, on a simpler molecule, to understand what it learns, how it is evaluated and where it can fail.

**The task:** given the positions of the 9 atoms of an ethanol molecule, predict its DFT energy.

| Part | Content | Course link | Time |
|---|---|---|---|
| 0 | Setup | | 5 min |
| 1 | Explore the dataset | C1 — watch your data | 40 min |
| 2 | Descriptors: samples, features x, label y | C1 — samples, features, labels | 25 min |
| 3 | Gradient descent by hand | C2 — optimisation, loss function | 25 min |
| 4 | Training, validation and test sets | C1 — how to split the data | 15 min |
| 5 | Models: from one neuron to a dense network | C2 — loss vs metric; C3 — DNN, training | 35 min |
| 6 | Experiments: data size and architecture | C1 — under/overfit; C3 — hyperparameters | 25 min |
| ⏱ | *Optional, if time allows:* learning rate (6.3), cross-validation (6.4), invariance test (7) | C1, C3 | 25 min |

**How to work**
- Run the cells in order (`Shift + Enter`).
- ✏️ **TODO** cells need one or two lines of code from you.
- 🔎 **Check cells** come right after a TODO: they only verify your code and print ✅ or ❌.
- C1, C2, C3 refer to the slides of AI courses 1, 2 and 3.

## Part 0 — Setup (5 min)

The first cell is only needed when the notebook runs in **JupyterLite** (Python inside the web browser): it installs the 3D viewer. Elsewhere it does nothing. In JupyterLite, the first run of this part can take up to a minute while Python starts in the browser.

In [ ]:
import sys
if sys.platform == "emscripten":          # True only in JupyterLite (Python running in the browser)
    import piplite
    await piplite.install("py3Dmol")

In [ ]:
import os
os.environ["OMP_NUM_THREADS"] = "2"   # share the server fairly: 2 CPU cores per student

import time
import warnings
import numpy as np
import matplotlib.pyplot as plt
import py3Dmol                         # interactive 3D molecule viewer
from scipy.spatial.distance import pdist
from sklearn.linear_model import LinearRegression
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import mean_absolute_error
from sklearn.exceptions import ConvergenceWarning
warnings.filterwarnings("ignore", category=ConvergenceWarning)   # some experiments stop training on purpose

rng = np.random.default_rng(0)        # fixed random seed: everybody gets the same results
DATA_PATH = "rmd17_ethanol.npz"       # full rMD17 ethanol file
SPLIT_DIR = "splits"                  # folder with the official rMD17 split files
SPLIT_ID  = 1                         # official split number (1 to 5)

def check(ok, msg_ok="✅ Correct!", msg_ko="❌ Not yet — read the hint and try again."):
    print(msg_ok if ok else msg_ko)

print("Ready.")

The next cell defines the functions that **draw the models** you will build. Its code is hidden: just run it (click on "…" to see it if you are curious).

In [ ]:
# Drawing helpers: diagrams of the pipeline and of the models (no need to read this cell)
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch, Circle

def draw_network(layers, names=None, activation="ReLU", ax=None, title=None, max_show=9, compact=False):
    # Draw a dense network: one column per layer, one circle per neuron (large layers are shortened with ⋮)
    if ax is None:
        _, ax = plt.subplots(figsize=(2.2 * len(layers) + 0.5, 4.2))
    names = names or (["input x"] + [f"hidden {k}" for k in range(1, len(layers) - 1)] + ["output ŷ"])
    colors = ["#4C72B0"] + ["#55A868"] * (len(layers) - 2) + ["#C44E52"]
    columns = []
    for li, n in enumerate(layers):
        if n <= max_show:
            items = [("node", k) for k in range(n)]
        else:
            items = [("node", k) for k in range(4)] + [("gap", None)] + [("node", n - 4 + k) for k in range(4)]
        ys = (np.arange(len(items))[::-1] - (len(items) - 1) / 2) * 0.55
        columns.append([(li * 1.6, y, kind) for (kind, _), y in zip(items, ys)])
    for left, right in zip(columns[:-1], columns[1:]):          # connections (one weight each)
        for (x0, y0, k0) in left:
            for (x1, y1, k1) in right:
                if k0 == "node" and k1 == "node":
                    ax.plot([x0, x1], [y0, y1], color="0.6", lw=0.6, alpha=0.6, zorder=1)
    for li, (col, n) in enumerate(zip(columns, layers)):
        for (x, y, kind) in col:
            if kind == "node":
                ax.add_patch(Circle((x, y), 0.17, facecolor=colors[li], edgecolor="black", lw=0.8, zorder=2))
            else:
                ax.text(x, y, "⋮", ha="center", va="center", fontsize=16, zorder=2)
        if compact:
            label = "x\n36" if li == 0 else ("ŷ" if li == len(layers) - 1 else f"{n}\n{activation}")
        else:
            label = f"{names[li]}\n{n} {'value' if li == 0 else 'neuron'}{'s' if n > 1 else ''}"
            if 0 < li < len(layers) - 1 and activation:
                label += f"\n+ {activation}"
        ax.text(col[0][0], -2.6, label, ha="center", va="top", fontsize=9)
    ax.set_xlim(-0.8, (len(layers) - 1) * 1.6 + 0.8); ax.set_ylim(-3.6, 2.6)
    ax.set_aspect("equal"); ax.axis("off")
    if title:
        ax.set_title(title, fontsize=10)
    return ax

def draw_single_neuron(x_label="x", ax=None, title=None):
    # The model ŷ = w0 + w1·x drawn as one neuron without activation
    if ax is None:
        _, ax = plt.subplots(figsize=(5.5, 2.4))
    ax.add_patch(Circle((0, 0), 0.25, facecolor="#4C72B0", edgecolor="black", zorder=2))
    ax.add_patch(Circle((0, 1.0), 0.18, facecolor="white", edgecolor="black", zorder=2))
    ax.add_patch(Circle((2.2, 0.4), 0.32, facecolor="#C44E52", edgecolor="black", zorder=2))
    ax.text(0, 1.0, "1", ha="center", va="center", fontsize=9)
    ax.text(2.2, 0.4, "Σ", ha="center", va="center", fontsize=13, color="white")
    ax.annotate("", (1.9, 0.33), (0.25, 0.0), arrowprops=dict(arrowstyle="->", lw=1.2))
    ax.annotate("", (1.92, 0.52), (0.18, 1.0), arrowprops=dict(arrowstyle="->", lw=1.2))
    ax.text(1.05, 0.0, "× w₁", fontsize=10, ha="center", va="top")
    ax.text(1.05, 0.95, "× w₀  (bias)", fontsize=10, ha="center", va="bottom")
    ax.annotate("", (3.4, 0.4), (2.52, 0.4), arrowprops=dict(arrowstyle="->", lw=1.2))
    ax.text(3.5, 0.4, "ŷ = w₀ + w₁·x", va="center", fontsize=11)
    ax.text(0, -0.45, x_label, ha="center", va="top", fontsize=9)
    ax.set_xlim(-0.8, 5.6); ax.set_ylim(-1.0, 1.5); ax.set_aspect("equal"); ax.axis("off")
    if title:
        ax.set_title(title, fontsize=10)
    return ax

def draw_pipeline(model_label="model\nŷ = f(x, w)", x_label="descriptor\nx = 36 values 1/r", ax=None):
    # The whole supervised-learning loop: geometry -> x -> model -> ŷ, compared with y; the error updates w
    if ax is None:
        _, ax = plt.subplots(figsize=(12, 3.4))
    boxes = {"geom": (0.0, 1.0, "geometry\n9 atoms × 3 coordinates", "#E8E8E8"),
             "x":    (2.6, 1.0, x_label, "#DCE6F2"),
             "f":    (5.2, 1.0, model_label, "#DDEED9"),
             "yhat": (7.8, 1.0, "prediction ŷ\n(kcal/mol)", "#F6DADA"),
             "y":    (7.8, -0.6, "label y\nDFT energy (kcal/mol)", "#F6DADA"),
             "loss": (10.4, 0.2, "error\nloss L(ŷ, y)", "#FFF2CC")}
    w, h = 2.0, 0.9
    for key, (x, y, text, color) in boxes.items():
        ax.add_patch(FancyBboxPatch((x - w / 2, y - h / 2), w, h, boxstyle="round,pad=0.05", facecolor=color, edgecolor="0.3"))
        ax.text(x, y, text, ha="center", va="center", fontsize=9)
    def arrow(p, q, **kw):
        ax.add_patch(FancyArrowPatch(p, q, arrowstyle="-|>", mutation_scale=14, lw=1.3, color="0.25", **kw))
    arrow((1.0, 1.0), (1.6, 1.0)); arrow((3.6, 1.0), (4.2, 1.0)); arrow((6.2, 1.0), (6.8, 1.0))
    arrow((8.8, 0.9), (9.4, 0.4)); arrow((8.8, -0.5), (9.4, 0.0))
    arrow((10.4, 0.7), (5.4, 1.5), connectionstyle="arc3,rad=0.35", linestyle="--")
    ax.text(8.6, 2.15, "training: change the weights w to reduce the error (gradient descent)", ha="center", fontsize=9, style="italic")
    ax.set_xlim(-1.1, 11.6); ax.set_ylim(-1.2, 2.5); ax.set_aspect("equal"); ax.axis("off")
    return ax

def count_weights(layers):
    # one weight per connection + one bias per neuron (course 3)
    return sum((a + 1) * b for a, b in zip(layers[:-1], layers[1:]))

def draw_architectures(architectures, n_inputs=36):
    # Small multiples: one network per architecture, panel width proportional to its number of layers
    layer_lists = [[n_inputs, *hidden, 1] for hidden in architectures]
    fig, axes = plt.subplots(1, len(layer_lists), figsize=(1.15 * sum(len(l) for l in layer_lists) + 1, 3.4),
                             gridspec_kw={"width_ratios": [len(l) for l in layer_lists]}, squeeze=False)
    for ax, hidden, layers in zip(axes[0], architectures, layer_lists):
        draw_network(layers, ax=ax, compact=True, title=f"{hidden}\n" + f"{count_weights(layers):,}".replace(",", " ") + " weights")
    plt.tight_layout(); plt.show()

print("Drawing functions ready.")

## Part 1 — Explore the dataset (40 min)

*C1 — watch your data.* In machine learning, understanding the dataset is usually the hardest and most important step: what was measured or computed, how, with which uncertainty, and how the samples relate to each other.

### 1.1 Where do the data come from?
**MD17** ([Chmiela *et al.*, *Sci. Adv.* 2017](https://doi.org/10.1126/sciadv.1603015)) contains *ab initio* molecular-dynamics trajectories of 10 small organic molecules in vacuum (benzene, ethanol, aspirin…). At each time step, DFT gives the energy of the molecule and the forces on its atoms; the forces move the atoms to the next step.

**rMD17**, the *revised* MD17 ([Christensen & von Lilienfeld, *Mach. Learn.: Sci. Technol.* 2020](https://doi.org/10.1088/2632-2153/abba6f)), takes **100 000 snapshots** per molecule from these trajectories and **recomputes** their energies and forces with a single, well-documented DFT setup (PBE functional, def2-SVP basis set, tight numerical settings), because the original labels were found to be numerically noisy.

- Download: [rMD17 on figshare](https://figshare.com/articles/dataset/Revised_MD17_dataset_rMD17_/12672038) — licence CC0 (free reuse).
- Structure of the archive:
```
rmd17/
├── npz_data/      rmd17_ethanol.npz, rmd17_aspirin.npz, …   one file per molecule
└── splits/        index_train_01.csv … index_train_05.csv   5 official training sets (1 000 snapshots each)
                   index_test_01.csv  … index_test_05.csv    5 official test sets
```

A **snapshot** is one geometry of the molecule at one instant of the simulation. Snapshots close in time are similar: keep this in mind for the whole session.

This is exactly the kind of data used to train a DNNF: geometries as input, DFT energies and forces as labels.

### 1.2 What is an `.npz` file?
A **zip archive of NumPy arrays**, each stored under a name (a *key*). `np.load` gives a dictionary-like object:
- `data.files` → list of keys
- `data["key"]` → the corresponding array

In [ ]:
data = np.load(DATA_PATH)
print(data.files)
for key in data.files:
    print(f"{key:16s} shape={str(data[key].shape):14s} dtype={data[key].dtype}")

The **first axis is always the snapshot index** (except for `nuclear_charges`, shared by all snapshots):

| key | shape | meaning | unit |
|---|---|---|---|
| `nuclear_charges` | (9,) | atomic number Z of each atom, in a fixed order | – |
| `coords` | (N, 9, 3) | snapshot × atom × (x, y, z) | Å |
| `energies` | (N,) | rMD17 DFT energy of each snapshot | kcal/mol |
| `forces` | (N, 9, 3) | rMD17 DFT force on each atom | kcal/mol/Å |
| `old_indices` | (N,) | position of the snapshot in the original MD17 trajectory (= time order) | – |
| `old_energies`, `old_forces` | (N,), (N, 9, 3) | original MD17 labels of the same snapshots | kcal/mol, kcal/mol/Å |

✏️ **TODO 1** — store the energies and the forces in the variables `E` and `F`.

In [ ]:
Z       = data["nuclear_charges"]
R       = data["coords"]
t_index = data["old_indices"]

E = data["energies"]
F = data["forces"]

🔎 **Check cell — TODO 1.** Run it to verify your code above. If it prints ❌, correct your code, run your cell again, then this one.

In [ ]:
check(E is not None and F is not None and E.shape == (len(R),) and F.shape == R.shape)

In [ ]:
N, n_atoms, _ = R.shape
symbols = np.array([{1: "H", 6: "C", 8: "O"}[z] for z in Z])
labels = [f"{s}{k}" for k, s in enumerate(symbols)]
print(f"{N} snapshots of a molecule with {n_atoms} atoms:", " ".join(labels))

### 1.3 One molecule
`R[i]` is a 9 × 3 table: the coordinates of molecule *i*. The cell below computes all interatomic distances and draws a bond when two atoms are closer than 1.6 Å.

In [ ]:
i = 0
D = np.linalg.norm(R[i][:, None, :] - R[i][None, :, :], axis=-1)   # 9 x 9 distance matrix

fig = plt.figure(figsize=(11, 4.5))
ax = fig.add_subplot(1, 2, 1)
im = ax.imshow(D, cmap="viridis")
ax.set_xticks(range(n_atoms), labels); ax.set_yticks(range(n_atoms), labels)
for a in range(n_atoms):
    for b in range(n_atoms):
        ax.text(b, a, f"{D[a, b]:.1f}", ha="center", va="center", fontsize=7, color="w")
fig.colorbar(im, ax=ax, label="distance (Å)")
ax.set_title(f"Interatomic distances, snapshot {i}")

ax = fig.add_subplot(1, 2, 2, projection="3d")
colors = {"C": "black", "O": "red", "H": "lightgray"}
ax.scatter(*R[i].T, c=[colors[s] for s in symbols], s=[300 if s != "H" else 120 for s in symbols], edgecolors="k")
for a in range(n_atoms):
    for b in range(a + 1, n_atoms):
        if D[a, b] < 1.6:
            ax.plot(*R[i][[a, b]].T, color="gray")
    ax.text(*R[i][a], labels[a])
ax.set_title("3D view")
plt.tight_layout(); plt.show()

### 1.4 The molecule in motion
The interactive view below plays 50 **consecutive** snapshots of the trajectory as a movie. Rotate with the mouse, zoom with the wheel.

In [ ]:
def to_xyz(coords):
    # one snapshot in the standard XYZ text format: number of atoms, comment line, one line per atom
    lines = [f"{s} {x:.4f} {y:.4f} {z:.4f}" for s, (x, y, z) in zip(symbols, coords)]
    return f"{n_atoms}\n\n" + "\n".join(lines) + "\n"

first_frames = np.argsort(t_index)[:50]          # 50 first snapshots in time order
view = py3Dmol.view(width=450, height=350)
view.addModelsAsFrames("".join(to_xyz(R[k]) for k in first_frames), "xyz")
view.setStyle({"stick": {}, "sphere": {"scale": 0.25}})
view.animate({"loop": "forward", "interval": 100})
view.zoomTo()
view.show()

### 1.5 Energies along the trajectory
Absolute DFT energies are large (≈ −97 000 kcal/mol); only their **variations** between geometries matter. We subtract the mean.

✏️ **TODO 2** — compute `E_rel`, the energies minus their mean value (`E.mean()`).

In [ ]:
E_rel = E - E.mean()

🔎 **Check cell — TODO 2.** Run it to verify your code above. If it prints ❌, correct your code, run your cell again, then this one.

In [ ]:
check(E_rel is not None and abs(E_rel.mean()) < 1e-6)

In [ ]:
order = np.argsort(t_index)          # snapshot indices sorted in time
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
axes[0].hist(E_rel, bins=60)
axes[0].set_xlabel("E − <E> (kcal/mol)"); axes[0].set_ylabel("count")
axes[0].set_title(f"Energy distribution (std = {E_rel.std():.1f} kcal/mol)")
axes[1].plot(t_index[order][:300], E_rel[order][:300], ".-", ms=3)
axes[1].set_xlabel("trajectory index (time)"); axes[1].set_ylabel("E − <E> (kcal/mol)")
axes[1].set_title("First 300 snapshots in time order")
plt.tight_layout(); plt.show()

### 1.6 Forces
The force on an atom is minus the derivative of the energy with respect to its position: $\vec F_a = -\partial E / \partial \vec r_a$.

In [ ]:
print("Largest |sum of the 9 forces| over all snapshots:", np.abs(F.sum(axis=1)).max(), "kcal/mol/Å")
print("Typical force component:", np.abs(F).mean().round(1), "kcal/mol/Å")

### 1.7 How reliable are the labels?
*C1 — ground truth.* The labels are not "the truth": they are the result of a calculation with a given method. The file contains two sets of labels for the **same geometries**: the original MD17 energies and the recomputed rMD17 energies. Let us compare them (each set relative to its own mean).

In [ ]:
E_old_rel = data["old_energies"] - data["old_energies"].mean()
diff = E_rel - E_old_rel

fig, axes = plt.subplots(1, 2, figsize=(10, 3.8))
sel = rng.choice(N, 3000, replace=False)
axes[0].scatter(E_old_rel[sel], E_rel[sel], s=3, alpha=0.4)
lims = [E_rel.min(), E_rel.max()]
axes[0].plot(lims, lims, "k--", lw=1)
axes[0].set(xlabel="MD17 energy − mean (kcal/mol)", ylabel="rMD17 energy − mean (kcal/mol)",
            title="Same geometries, two sets of labels")
axes[1].hist(diff, bins=60)
axes[1].set(xlabel="rMD17 − MD17 (kcal/mol)", ylabel="count",
            title=f"Difference: std = {diff.std():.2f} kcal/mol")
plt.tight_layout(); plt.show()

## Part 2 — Descriptors: samples, features x and label y (25 min)

*C1 — samples, features and labels.* In our dataset:
- a **sample** is one snapshot (one geometry of the molecule);
- its **features** are the numbers that describe it — to be chosen;
- its **label** is what we want to predict: the DFT energy.

The model needs **the same number of features for every sample**. How do we turn a molecule into a vector of numbers?

### 2.1 Two possible descriptors
**Option A — Cartesian coordinates.** Flatten the 9 × 3 table into 27 numbers.

In [ ]:
X_cart = R.reshape(N, -1)
print(X_cart.shape)

**Option B — inverse distances.** For each pair of atoms (a, b), compute $1/r_{ab}$. With 9 atoms there are 9 × 8 / 2 = 36 pairs.
Close atoms give large values, as interactions are strongest at short range.

How the code below works:
- `ia` and `ib` list the two atoms of each of the 36 pairs: pair 0 is (atom 0, atom 1), pair 1 is (atom 0, atom 2)… pair 35 is (atom 7, atom 8).
- `coords[:, ia, :] - coords[:, ib, :]` gives, for every snapshot at once, the 36 vectors between paired atoms: shape (N, 36, 3).
- You must turn each vector (3 numbers) into its length, then take the inverse: shape (N, 36).

✏️ **TODO 3** — complete the function.
*Hint:* the length of a vector is `np.linalg.norm(vector, axis=-1)` (`axis=-1` = along the last axis, the x, y, z one).

<details>
<summary>💡 <b>Hint: the same computation written with loops</b> (click to open)</summary>

This loop version does exactly what the function must do, one snapshot and one pair at a time. It is easier to read but about 1 000 times slower — use it to understand, not in your answer.

```python
result = np.zeros((len(coords), 36))
for s in range(len(coords)):              # each snapshot
    for p in range(36):                   # each pair of atoms
        a, b = ia[p], ib[p]
        vector = coords[s, a] - coords[s, b]            # 3 numbers: x, y, z
        distance = np.sqrt((vector ** 2).sum())         # length of the vector
        result[s, p] = 1 / distance
```
</details>

In [ ]:
ia, ib = np.triu_indices(n_atoms, k=1)     # the 36 pairs (a < b)

def inverse_distances(coords):
    # coords: (n_snapshots, 9, 3)  ->  result: (n_snapshots, 36)
    vectors = coords[:, ia, :] - coords[:, ib, :]     # (n_snapshots, 36, 3)
    distances = np.linalg.norm(vectors, axis=-1)      # (n_snapshots, 36)
    return 1.0 / distances                            # (n_snapshots, 36)

🔎 **Check cell — TODO 3.** Run it to verify your code above. If it prints ❌, correct your code, run your cell again, then this one.

In [ ]:
X_inv = inverse_distances(R)
check(X_inv is not None and X_inv.shape == (N, 36) and np.allclose(X_inv[0], 1 / pdist(R[0])))

**Does the description depend on where the molecule is, or how it is oriented?** Let us move and rotate one molecule and compare both descriptors.

In [ ]:
def random_rotation(rng):
    q, r = np.linalg.qr(rng.normal(size=(3, 3)))
    q *= np.sign(np.diag(r))
    if np.linalg.det(q) < 0:
        q[:, 0] *= -1
    return q

Rot = random_rotation(rng)
mol = R[:1]                                    # snapshot 0, shape (1, 9, 3)
mol_moved = mol @ Rot.T + np.array([5.0, -2.0, 1.0])   # rotated, then translated by 5.5 Å

change_cart = np.abs(mol_moved.reshape(1, -1) - mol.reshape(1, -1)).max()
change_inv  = np.abs(inverse_distances(mol_moved) - inverse_distances(mol)).max()
print(f"Cartesian coordinates: largest change = {change_cart:.3f} Å")
print(f"Inverse distances:     largest change = {change_inv:.1e} 1/Å")

### 2.2 The notation of the course: x, y, ŷ, w
From now on we use the notation of course 2:

| symbol | meaning | here | shape |
|---|---|---|---|
| **x** | input of the model = features of one sample | 36 inverse distances of one snapshot | (36,) |
| **X** | all inputs stacked, one row per sample | `X_inv` | (N, 36) |
| **y** | target = label of one sample | DFT energy of that snapshot, E − ⟨E⟩ | scalar |
| **ŷ** | prediction of the model for that sample | predicted energy | scalar |
| **w** | weights (parameters) of the model, learned from the data | — | depends on the model |

A model is a function $\hat y = f(\mathbf x, \mathbf w)$. **Training** = finding the weights **w** that make ŷ close to y on the training samples. **Inference** = using the trained model on new samples.

The whole method in one picture: what you will build during this session.

In [ ]:
draw_pipeline()
plt.show()

In [ ]:
y = E_rel                                  # labels of all samples

k = 0                                      # look at one sample (one snapshot)
print(f"Sample {k}\n")
print(f"x = its {X_inv.shape[1]} features (first 8 shown):")
print(f"  {'feature':>7s}   {'atom pair':9s}  {'distance r (Å)':>14s}  {'x = 1/r (1/Å)':>13s}")
for j in range(8):
    pair = f"{labels[ia[j]]}–{labels[ib[j]]}"
    print(f"  {j:7d}   {pair:9s}  {1 / X_inv[k, j]:14.3f}  {X_inv[k, j]:13.3f}")
print("      ...")
position = "below" if y[k] < 0 else "above"
print(f"\ny = its label = E − <E> = {y[k]:.2f} kcal/mol   "
      f"(this geometry is {abs(y[k]):.1f} kcal/mol {position} the average energy)")

fig, axes = plt.subplots(1, 2, figsize=(11, 3.2), gridspec_kw={"width_ratios": [12, 1]})
im = axes[0].imshow(X_inv[:15], aspect="auto", cmap="viridis")
axes[0].set_xlabel("feature index (atom pair)"); axes[0].set_ylabel("sample (snapshot)")
axes[0].set_title("X : 15 first rows (one row = one sample x)")
fig.colorbar(im, ax=axes[0], label="1/r (1/Å)")
axes[1].imshow(y[:15, None], aspect="auto", cmap="coolwarm")
axes[1].set_xticks([]); axes[1].set_yticks([]); axes[1].set_title("y")
plt.tight_layout(); plt.show()

### 2.3 Which features carry information?
A feature that never changes cannot help the model distinguish two snapshots. Which atom pairs change most along the trajectory?

The 36 pairs fall into three families, depending on how many bonds separate the two atoms:

| family | example | changes when… |
|---|---|---|
| **bonded** (1 bond) | C0–O2 | a bond stretches |
| **1–3 pairs** (2 bonds, one atom in between) | H3–O2, via C0 | a bond angle bends |
| **distant pairs** (3 bonds or more) | H5–H8 | a group rotates around a bond (torsion) |

The cell below sorts the pairs by how much their **distance r** varies over all snapshots (standard deviation, in Å), coloured by family.

In [ ]:
pair_names = [f"{labels[a]}–{labels[b]}" for a, b in zip(ia, ib)]

bonded = (D < 1.6) & ~np.eye(n_atoms, dtype=bool)                    # bonds of snapshot 0
one_three = ((bonded.astype(int) @ bonded.astype(int)) > 0) & ~bonded & ~np.eye(n_atoms, dtype=bool)
family = np.where(bonded[ia, ib], "bonded", np.where(one_three[ia, ib], "1–3 pair", "distant pair"))
fam_color = {"bonded": "tab:orange", "1–3 pair": "tab:green", "distant pair": "tab:blue"}

spread = (1.0 / X_inv).std(axis=0)              # variation of each distance r over all samples
rank = np.argsort(spread)[::-1]                 # from most to least variable

plt.figure(figsize=(12, 3.8))
plt.bar(range(36), spread[rank], color=[fam_color[family[j]] for j in rank])
plt.xticks(range(36), [pair_names[j] for j in rank], rotation=90, fontsize=8)
for name, c in fam_color.items():
    plt.bar(0, 0, color=c, label=name)
plt.legend()
plt.ylabel("std of r over samples (Å)")
plt.title("How much each interatomic distance varies along the trajectory")
plt.tight_layout(); plt.show()

## Part 3 — Gradient descent by hand (25 min)

*C2 — optimisation problem.* In course 2 you saw gradient descent used to find the geometry of minimum energy:
the loss is the energy E, the variables are the atom coordinates, and each step moves the atoms downhill:
$X_{n+1} = X_n - \eta\,\nabla E(X_n)$.

Training a model is **the same algorithm with different variables**: the loss is the prediction error $L(\mathbf w)$, the variables are the **weights**, and each step changes the weights downhill:

$$\mathbf w_{n+1} = \mathbf w_n - \eta\,\nabla L(\mathbf w_n)$$

η is the **learning rate**. To *see* it, we use a model with only two weights.

### 3.1 A one-feature model with a physical meaning
In the harmonic approximation, stretching a bond by Δr costs an energy proportional to (Δr)². So we build **one feature per snapshot**:

$$x = \sum_{\text{bonds}} (\Delta r)^2$$

the sum of the squared bond stretches (Δr = bond length minus its mean value), and the model

$$\hat y = w_0 + w_1\, x$$

$w_1$ plays the role of an average bond "spring constant". We use 1 000 random snapshots.

In [ ]:
draw_single_neuron("x = Σ (Δr)²", title="The smallest possible model: one input, two weights")
plt.show()

In [ ]:
distances = 1.0 / X_inv                       # (N, 36): all interatomic distances, in Å
is_bond = family == "bonded"                  # the 8 bonded pairs (Part 2.3)
stretch = distances[:, is_bond] - distances[:, is_bond].mean(axis=0)
x_raw = (stretch ** 2).sum(axis=1)            # Σ (Δr)², in Å²

gd_idx = rng.choice(N, 1000, replace=False)
x_scale = x_raw[gd_idx].std()
x_gd = x_raw[gd_idx] / x_scale                # divided by its standard deviation: values of order 1
y_gd = y[gd_idx]

plt.figure(figsize=(5, 3.5))
plt.scatter(x_gd, y_gd, s=4, alpha=0.4, color="gray")
plt.xlabel("x = Σ (Δr)²   (scaled, no unit)"); plt.ylabel("y = E − <E>  (kcal/mol)")
plt.title(f"correlation = {np.corrcoef(x_gd, y_gd)[0, 1]:.2f}")
plt.show()

### 3.2 Loss and gradient
The loss is the **mean squared error** (C2):

$$L(w_0, w_1) = \frac{1}{n}\sum_i (\hat y_i - y_i)^2$$

Its derivatives with respect to each weight (the gradient) are:

$$\frac{\partial L}{\partial w_0} = \frac{2}{n}\sum_i (\hat y_i - y_i), \qquad \frac{\partial L}{\partial w_1} = \frac{2}{n}\sum_i (\hat y_i - y_i)\,x_i$$

✏️ **TODO 4** — write the gradient-descent step inside the loop: the new `w` is the old `w` minus `eta` times `gradient(w)`.

In [ ]:
def predict(w, x):
    return w[0] + w[1] * x

def loss(w):
    return np.mean((predict(w, x_gd) - y_gd) ** 2)

def gradient(w):
    error = predict(w, x_gd) - y_gd
    return np.array([2 * error.mean(), 2 * (error * x_gd).mean()])

def gradient_descent(eta, w_start, n_steps=40):
    w = np.array(w_start, dtype=float)
    path = [w.copy()]
    for n in range(n_steps):
        w = w - eta * gradient(w)
        path.append(w.copy())
    return np.array(path)

🔎 **Check cell — TODO 4.** Run it to verify your code above. If it prints ❌, correct your code, run your cell again, then this one.

In [ ]:
try:
    ok = np.allclose(gradient_descent(0.1, [0, 0], n_steps=1)[1], -0.1 * gradient(np.zeros(2)))
except Exception:
    ok = False
check(ok)

### 3.3 Watch the descent
We start from the same wrong weights and run 40 steps with three learning rates.
- **Left:** the data and the model line at several iterations.
- **Middle:** the loss as a landscape over $(w_0, w_1)$ — the lines are contours of equal loss, the star is the minimum.
- **Right:** the loss at each iteration.

In [ ]:
w_best = np.polyfit(x_gd, y_gd, 1)[::-1]          # exact minimum, for reference
w_start = w_best + np.array([11.0, -7.0])
etas = [0.02, 0.15, 0.2]                            # try other values!

# loss landscape on a grid of (w0, w1)
W0, W1 = np.meshgrid(np.linspace(w_best[0] - 11, w_best[0] + 15, 200),
                     np.linspace(w_best[1] - 9,  w_best[1] + 7,  200))
# L = mean((w0 + w1*x - y)^2), expanded so that it is computed for the whole grid at once
mx, my, mxx, mxy, myy = x_gd.mean(), y_gd.mean(), (x_gd**2).mean(), (x_gd*y_gd).mean(), (y_gd**2).mean()
L_grid = W0**2 + W1**2 * mxx + myy + 2*W0*W1*mx - 2*W0*my - 2*W1*mxy

fig, ax = plt.subplots(1, 3, figsize=(15, 4.3))
colors = ["tab:blue", "tab:green", "tab:red"]
paths = {eta: gradient_descent(eta, w_start) for eta in etas}

ax[0].scatter(x_gd, y_gd, s=4, alpha=0.3, color="gray")
xs = np.array([x_gd.min(), x_gd.max()])
for it, c in zip([0, 2, 10, 40], plt.cm.viridis(np.linspace(0, 0.9, 4))):
    w = paths[etas[1]][it]
    ax[0].plot(xs, predict(w, xs), color=c, label=f"iteration {it}")
ax[0].set(xlabel="x (scaled)", ylabel="y (kcal/mol)", title=f"Model ŷ = w₀ + w₁·x  (η = {etas[1]})")
ax[0].legend(fontsize=8)

ax[1].contour(W0, W1, L_grid, levels=np.geomspace(L_grid.min() * 1.02, L_grid.max(), 15),
              colors="0.6", linewidths=0.8)
for eta, c in zip(etas, colors):
    p = paths[eta][:12] if loss(paths[eta][-1]) > loss(w_start) else paths[eta]
    ax[1].plot(p[:, 0], p[:, 1], "o-", ms=3, color=c, label=f"η = {eta}")
    ax[2].semilogy([loss(w) for w in paths[eta]], "o-", ms=3, color=c, label=f"η = {eta}")
ax[1].plot(*w_best, "k*", ms=14, label="minimum")
ax[1].set_xlim(W0.min(), W0.max()); ax[1].set_ylim(W1.min(), W1.max())
ax[1].set(xlabel="w₀ (kcal/mol)", ylabel="w₁ (kcal/mol)", title="Loss landscape L(w₀, w₁) and descent paths")
ax[1].legend(fontsize=8)
ax[2].set_ylim(loss(w_best) * 0.8, loss(w_start) * 100)
ax[2].set(xlabel="iteration n", ylabel="L  (kcal/mol)²", title="Loss along the descent")
ax[2].legend(fontsize=8)
plt.tight_layout(); plt.show()

Gradient descent and the exact solution (least squares) should agree after enough steps:

In [ ]:
w_gd = gradient_descent(etas[1], w_start, n_steps=2000)[-1]
print("gradient descent (2000 steps):", np.round(w_gd, 3))
print("exact minimum                :", np.round(w_best, 3))
print(f"w1 in physical units: {w_gd[1] / x_scale:.0f} kcal/mol/Å²")

## Part 4 — Training, validation and test sets (15 min)

*C1 — how to split the data.*
- **Training set:** used to learn the weights.
- **Validation set:** used to *choose* between models and hyperparameters (number of layers, learning rate…).
- **Test set:** emulates the "outer world". **Never seen during training, and never used to make choices** — only for the final evaluation.

### 4.1 The official rMD17 split
The authors of rMD17 provide 5 fixed splits (files `index_train_0X.csv` and `index_test_0X.csv`), each listing snapshot numbers. Everybody who publishes on rMD17 uses them, so results can be compared between papers.
They also warn: **do not train on more than 1 000 snapshots** of this dataset — the snapshots come from one MD trajectory and are not independent.

In [ ]:
def load_split(kind, split_id):
    # kind is "train" or "test"; returns an array of snapshot indices
    return np.loadtxt(f"{SPLIT_DIR}/index_{kind}_0{split_id}.csv", dtype=int)

train_off = load_split("train", SPLIT_ID)
test_off  = load_split("test",  SPLIT_ID)

# validation set: we keep 20 % of the official training set aside
n_val = len(train_off) // 5
train_fit, val = train_off[:-n_val], train_off[-n_val:]

print(f"official split {SPLIT_ID}: {len(train_off)} training ({len(train_fit)} fit + {len(val)} validation), "
      f"{len(test_off)} test snapshots")
print("Snapshots in both training and test sets:", len(np.intersect1d(train_off, test_off)))

### 4.2 A harder test: the time-block split
The official split draws snapshots at random along the trajectory. Here is an alternative: train on the beginning of the trajectory, test on the end — closer to "predicting configurations the model has never explored". We keep the same sizes as the official split.

✏️ **TODO 5** — `order` contains the snapshot indices sorted in time. Define `train_pool` as the first 70 % of `order` and `test_pool` as the last 30 %.
*Hint:* `order[:cut]` and `order[cut:]`.

In [ ]:
cut = int(0.7 * N)
train_pool = order[:cut]
test_pool  = order[cut:]

🔎 **Check cell — TODO 5.** Run it to verify your code above. If it prints ❌, correct your code, run your cell again, then this one.

In [ ]:
check(train_pool is not None and test_pool is not None
      and t_index[train_pool].max() < t_index[test_pool].min() and len(train_pool) + len(test_pool) == N)

In [ ]:
train_time = rng.choice(train_pool, len(train_off), replace=False)
test_time  = rng.choice(test_pool,  len(test_off),  replace=False)

## Part 5 — Models: from one neuron to a dense network (35 min)

### 5.1 Three models
| model | equation | weights w |
|---|---|---|
| **Baseline** | $\hat y = \bar y$ (mean training energy) | none |
| **Linear regression** | $\hat y = w_0 + \sum_k w_k x_k$ | 1 + 36 = 37 |
| **Dense network (MLP)** | layers of neurons with a non-linear activation | many |

*C3 — the neuron.* A neuron computes a weighted sum of its inputs plus a bias, then applies an activation function $f$: $\;f(w_0 + \sum_k w_k x_k)$.
**Linear regression is a single neuron without activation.** Course 3 explained why the activation is essential: without it, stacking layers would still give a linear model.

The two models side by side. The **input layer** has one value per feature (its size is set by the descriptor), the **output layer** one neuron (we predict one value, the energy), and the **hidden layers** are chosen by trial and error. Each grey line is one weight. Large layers are shortened with ⋮.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.3), gridspec_kw={"width_ratios": [1, 2]})
draw_network([36, 1], names=["input x", "output ŷ"], ax=axes[0], title="Linear regression: one neuron, no activation")
draw_network([36, 64, 64, 1], ax=axes[1], title="Dense network (MLP): two hidden layers with ReLU")
plt.show()

### 5.2 Loss vs metric
*C2 — loss or metric?*
- The **loss** is what training minimises. Here: the **MSE** (scikit-learn uses MSE / 2), minimised by **Adam**, a refined gradient descent, on mini-batches of 200 samples (C3 — batch training).
- The **metric** is what *we* read to judge the model: the **MAE** in kcal/mol, easier to interpret. *Chemical accuracy* is about 1 kcal/mol.

✏️ **TODO 6** — complete `evaluate` so that it returns the MAE between the true energies `y_true` and the predictions `y_pred`.
*Hint:* the function `mean_absolute_error` is already imported.

In [ ]:
def make_mlp(hidden=(64, 64), learning_rate=0.001, max_iter=3000):
    return make_pipeline(StandardScaler(),
                         MLPRegressor(hidden_layer_sizes=hidden, learning_rate_init=learning_rate,
                                      max_iter=max_iter, random_state=0))

def evaluate(model, X, train, test):
    model.fit(X[train], y[train])
    y_true, y_pred = y[test], model.predict(X[test])
    return mean_absolute_error(y_true, y_pred)

🔎 **Check cell — TODO 6.** Run it to verify your code above. If it prints ❌, correct your code, run your cell again, then this one.

In [ ]:
check(evaluate(LinearRegression(), X_inv, train_off, test_off) is not None)

### 5.3 Compare all models (≈ 10 s)
`StandardScaler` rescales each feature to mean 0 and standard deviation 1 before the network — as we did by hand for x in Part 3.

In [ ]:
splits   = {"official": (train_off, test_off), "time": (train_time, test_time)}
features = {"cartesian": X_cart, "inverse dist.": X_inv}

print(f"{'split':8s} {'model':28s} test MAE (kcal/mol)")
for s_name, (tr, te) in splits.items():
    print(f"{s_name:8s} {'baseline (mean)':28s} {mean_absolute_error(y[te], np.full(len(te), y[tr].mean())):.2f}")
    for f_name, X in features.items():
        print(f"{s_name:8s} {'linear / ' + f_name:28s} {evaluate(LinearRegression(), X, tr, te):.2f}")
        print(f"{s_name:8s} {'MLP / ' + f_name:28s} {evaluate(make_mlp(), X, tr, te):.2f}")

### 5.4 Inside the training of the network
The reference model: MLP on inverse distances, official split. We count its weights and plot its loss during training — the same kind of curve as in Part 3, but with thousands of weights instead of 2.

In [ ]:
best = make_mlp()
mae = evaluate(best, X_inv, train_off, test_off)
mlp = best[-1]                                       # the network inside the pipeline

n_weights = sum(W.size for W in mlp.coefs_) + sum(b.size for b in mlp.intercepts_)
print(f"Number of weights: {n_weights}   (for {len(train_off)} training samples)")
print(f"Training stopped after {mlp.n_iter_} epochs (one epoch = one pass over the training set)")

fig, ax = plt.subplots(1, 2, figsize=(10, 4))
ax[0].semilogy(mlp.loss_curve_)
ax[0].set(xlabel="epoch", ylabel="training loss (MSE / 2), (kcal/mol)²", title="Loss during training (Adam)")
y_pred = best.predict(X_inv[test_off])
ax[1].scatter(y[test_off], y_pred, s=4, alpha=0.5)
lims = [y.min(), y.max()]
ax[1].plot(lims, lims, "k--", lw=1)
ax[1].set(xlabel="y = DFT energy (kcal/mol)", ylabel="ŷ = predicted energy (kcal/mol)",
          title=f"Parity plot on the test set — MAE = {mae:.2f} kcal/mol")
plt.tight_layout(); plt.show()

## Part 6 — Experiments (25 min, + optional parts)

*C3 — hyperparameters* are fixed before training and not learned: number of layers and neurons, learning rate η, batch size, optimiser… We choose them using the **validation set**, never the test set.

### 6.1 Learning curve: does more data help? (≈ 10 s)
We train on growing subsets of the official training set (at most 1 000 snapshots, as recommended) and evaluate on the test set.

In [ ]:
sizes = [50, 100, 250, 500, 1000]
maes = []
for n in sizes:
    t0 = time.time()
    maes.append(evaluate(make_mlp(), X_inv, train_off[:n], test_off))
    print(f"n_train = {n:5d}   MAE = {maes[-1]:.2f} kcal/mol   ({time.time() - t0:.1f} s)")

plt.figure(figsize=(5, 3.5))
plt.loglog(sizes, maes, "o-")
plt.axhline(1.0, color="gray", ls=":", label="chemical accuracy")
plt.xlabel("training set size"); plt.ylabel("test MAE (kcal/mol)"); plt.legend()
plt.grid(True, which="both", alpha=0.3)
plt.show()

### 6.2 Choosing the architecture
*C1 — underfit, right fit, overfit:*

| | training error | validation error |
|---|---|---|
| **underfit** | high | high, ≈ training error |
| **right fit** | low | low, slightly above training error |
| **overfit** | very low | much higher than training error |

✏️ **TODO 7** — try at least four architectures, e.g. `(8,)`, `(32,)`, `(64, 64)`, `(128, 128)`. Edit the list and run. Fill in the table below, including your diagnosis.

In [ ]:
architectures = [(8,), (32,), (64, 64), (128, 128)]
draw_architectures(architectures)       # one diagram per architecture

for hidden in architectures:
    t0 = time.time()
    model = make_mlp(hidden)
    mae_val = evaluate(model, X_inv, train_fit, val)
    mae_train = mean_absolute_error(y[train_fit], model.predict(X_inv[train_fit]))
    print(f"{str(hidden):12s} train MAE = {mae_train:.2f}   validation MAE = {mae_val:.2f} kcal/mol   ({time.time() - t0:.1f} s)")

| architecture | train MAE | validation MAE | diagnosis (underfit / right fit / overfit) |
|---|---|---|---|
| | | | |
| | | | |
| | | | |
| | | | |

Choose your architecture using the **validation** error, then evaluate it **once** on the test set:

In [ ]:
chosen = (64, 64)        # your choice
mae_test = evaluate(make_mlp(chosen), X_inv, train_off, test_off)
print(f"Chosen architecture {chosen}: test MAE = {mae_test:.2f} kcal/mol")

### ⏱ 6.3 Optional — the learning rate η (≈ 20 s)
Back to Part 3: η controls the size of each gradient step. Here we train the same network with four learning rates, for at most 300 epochs. Training also stops by itself when the loss no longer decreases.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
for lr in [1e-4, 1e-3, 1e-2, 1e-1]:
    model = make_mlp(chosen, learning_rate=lr, max_iter=300)
    mae_val = evaluate(model, X_inv, train_fit, val)
    ax.semilogy(model[-1].loss_curve_, label=f"η = {lr:g}   (validation MAE {mae_val:.2f})")
ax.set(xlabel="epoch", ylabel="training loss (MSE / 2), (kcal/mol)²", title="Effect of the learning rate")
ax.legend(fontsize=8)
plt.show()

### ⏱ 6.4 Optional — cross-validation with the 5 official splits (≈ 10 s)
*C1 — cross-validation.* One split gives one number; training on several splits shows how much the result depends on the split.

In [ ]:
cv_maes = []
for k in range(1, 6):
    tr, te = load_split("train", k), load_split("test", k)
    cv_maes.append(evaluate(make_mlp(chosen), X_inv, tr, te))
    print(f"split {k}: test MAE = {cv_maes[-1]:.2f} kcal/mol")
print(f"\nMAE = {np.mean(cv_maes):.2f} ± {np.std(cv_maes):.2f} kcal/mol")

## ⏱ Part 7 — Optional — Invariance test (10 min)

Rotate the whole test set. The true energies do not change. What do the models predict?

In [ ]:
R_rot = R[test_off] @ random_rotation(rng).T

mlp_cart = make_mlp(); evaluate(mlp_cart, X_cart, train_off, test_off)
for name, model, X_before, X_after in [
    ("MLP / cartesian",     mlp_cart, X_cart[test_off], R_rot.reshape(len(test_off), -1)),
    ("MLP / inverse dist.", best,     X_inv[test_off],  inverse_distances(R_rot)),
]:
    before = mean_absolute_error(y[test_off], model.predict(X_before))
    after  = mean_absolute_error(y[test_off], model.predict(X_after))
    print(f"{name:20s} MAE before rotation {before:5.2f}   after rotation {after:5.2f} kcal/mol")